## Installation

# TinyStories SLM Ablation Notebook

This notebook is organized into five sections:

1. Load and inspect the TinyStories dataset.
2. Train and profile custom tokenizers for the tokenizer-effect study.
3. Train the 7M GPT-style baseline across tokenizer variants.
4. Train the same 7M baseline across corpus sizes under constant compute.
5. Evaluate memorization on the 10k-subset model using `ngram_containment`.

Recommended execution order after a fresh Colab restart: installation, drive mount, shared setup cells, then Sections 1 through 5 in order.

In [ ]:
# %%
# Install all notebook dependencies in Colab.
# `accelerate` is required by the HuggingFace Trainer stack, and `wandb`
# is used for all experiment tracking requested in the ablation study.
!pip install -q datasets transformers sentencepiece tokenizers accelerate wandb

In [ ]:
# %%
# Mount Google Drive so trained tokenizers, packed datasets, and model checkpoints
# persist beyond the current Colab runtime session.
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %%
# Colab optimization: authenticate WandB once up front to avoid login prompts
# during training loops that can stall the notebook UI.
import wandb

wandb.login()

### Shared experiment setup

The next few cells define the common imports, paths, helper functions, and dataset handles used by every experiment section.

In [ ]:
# %%
# Shared imports, experiment constants, and dataset loading.
# This cell should be run once after installing dependencies and mounting Drive.

import os
import gc
import random
import warnings

import numpy as np
import torch
import wandb
from datasets import DatasetDict, load_dataset, load_from_disk
from tokenizers import ByteLevelBPETokenizer
from transformers import (
    DataCollatorForLanguageModeling,
    GPT2Config,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

# Reproducibility settings keep data sampling and model initialization stable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Core experiment settings used across all ablation runs.
vocab_sizes = [2048, 4096, 8192, 16384, 50257]
subset_sizes = [10_000, 50_000, 250_000, 1_000_000, 2_119_719]
block_size = 512

# Load the TinyStories dataset once and reuse it across all later cells.
# Colab optimization: avoid repeated dataset loads to reduce RAM pressure.
raw = load_dataset("roneneldan/TinyStories")
train_ds = raw["train"]
val_ds = raw["validation"] if "validation" in raw else train_ds.select(range(min(5000, len(train_ds))))

# Mutable lookup dictionaries are rebuilt during preprocessing stages.
trained_tokenizer_dirs = {}
corpus_subset_dirs = {}

# Colab optimization:
# - Keep durable artifacts on Drive.
# - Keep packed training datasets on local SSD to avoid Drive I/O bottlenecks.
DRIVE_ROOT = "/content/drive/MyDrive/llm_folder"
LOCAL_DATA_ROOT = "/content/local_data"

ABLATION_ROOT = os.path.join(DRIVE_ROOT, "ablation_artifacts")
TOKENIZER_ROOT = os.path.join(ABLATION_ROOT, "tokenizers")
PROFILE_ROOT = os.path.join(ABLATION_ROOT, "profiles")
MODEL_ROOT = os.path.join(ABLATION_ROOT, "models")
TOKENIZER_PACKED_ROOT = os.path.join(LOCAL_DATA_ROOT, "tokenizer_ablation_packed")
CORPUS_PACKED_ROOT = os.path.join(LOCAL_DATA_ROOT, "corpus_ablation_packed")

for path in [
    ABLATION_ROOT,
    TOKENIZER_ROOT,
    PROFILE_ROOT,
    MODEL_ROOT,
    TOKENIZER_PACKED_ROOT,
    CORPUS_PACKED_ROOT,
]:
    os.makedirs(path, exist_ok=True)

print(f"Train size: {len(train_ds):,}")
print(f"Validation size: {len(val_ds):,}")
print(f"Drive artifact root: {ABLATION_ROOT}")
print(f"Local packed-data root: {LOCAL_DATA_ROOT}")
print("Base imports and dataset handles are ready.")

In [ ]:
# %%
# Helper functions for memorization analysis and packed-dataset creation.
# Packing tokens into contiguous fixed-length blocks keeps training input format
# consistent across tokenizer and corpus-size experiments.

def get_ngrams(tokens, n):
    """Return the set of all token n-grams in a token sequence."""
    return set(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def ngram_containment(generated_text, tok, train_texts, n=8, max_train_docs=20000, seed=42):
    """Measure how much of a generated sample overlaps with training n-grams."""
    random.seed(seed)
    sample_texts = train_texts if len(train_texts) <= max_train_docs else random.sample(train_texts, max_train_docs)

    gen_ids = tok(generated_text, add_special_tokens=False)["input_ids"]
    if len(gen_ids) < n:
        return {"n": n, "containment": 0.0, "gen_ngrams": 0, "train_docs_used": len(sample_texts)}

    gen_ngrams = get_ngrams(gen_ids, n)
    train_ngrams = set()
    for text in sample_texts:
        ids = tok(text, add_special_tokens=False)["input_ids"]
        if len(ids) >= n:
            train_ngrams |= get_ngrams(ids, n)

    overlap = len(gen_ngrams & train_ngrams)
    return {
        "n": n,
        "containment": overlap / max(1, len(gen_ngrams)),
        "gen_ngrams": len(gen_ngrams),
        "train_ngrams": len(train_ngrams),
        "train_docs_used": len(sample_texts),
        "overlap_ngrams": overlap,
    }


def tokenize_and_pack_to_disk(train_subset, val_subset, tokenizer, out_dir, block_size=512):
    """Tokenize text, pack it into fixed-size blocks, and save the result to disk."""
    os.makedirs(out_dir, exist_ok=True)

    def tokenize_fn(batch):
        return tokenizer(batch["text"], add_special_tokens=False, return_attention_mask=False)

    # First tokenize each example independently.
    tok_train = train_subset.map(tokenize_fn, batched=True, remove_columns=train_subset.column_names)
    tok_val = val_subset.map(tokenize_fn, batched=True, remove_columns=val_subset.column_names)

    def group_texts(examples):
        # Concatenate token streams so the language model sees contiguous blocks.
        concatenated = []
        for ids in examples["input_ids"]:
            concatenated.extend(ids)

        total_len = (len(concatenated) // block_size) * block_size
        concatenated = concatenated[:total_len]
        input_ids = [concatenated[i:i + block_size] for i in range(0, total_len, block_size)]
        return {"input_ids": input_ids, "labels": input_ids.copy()}

    lm_train = tok_train.map(group_texts, batched=True, remove_columns=tok_train.column_names)
    lm_val = tok_val.map(group_texts, batched=True, remove_columns=tok_val.column_names)

    packed = DatasetDict({"train": lm_train, "validation": lm_val})
    packed.save_to_disk(out_dir)

    # Free intermediate objects to reduce Colab RAM pressure.
    del tok_train, tok_val, lm_train, lm_val, packed
    gc.collect()
    return out_dir

In [ ]:
# %%
# Helper functions for model construction and tokenizer loading.
# These functions preserve your baseline 7M architecture while allowing
# repeated training under different tokenizer settings.

def build_baseline_model(tokenizer):
    """Create the baseline GPT-style model used throughout all ablations."""
    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        n_positions=block_size,
        n_ctx=block_size,
        n_embd=256,
        n_layer=6,
        n_head=8,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    return GPT2LMHeadModel(config)


def load_fast_tokenizer_from_folder(tokenizer_dir):
    """Load a saved byte-level BPE tokenizer as a HuggingFace fast tokenizer."""
    vocab_path = os.path.join(tokenizer_dir, "vocab.json")
    merges_path = os.path.join(tokenizer_dir, "merges.txt")
    tokenizer_json = os.path.join(tokenizer_dir, "tokenizer.json")

    if not os.path.exists(vocab_path) or not os.path.exists(merges_path):
        raise FileNotFoundError(f"Missing vocab.json or merges.txt in {tokenizer_dir}")

    bpe = ByteLevelBPETokenizer(vocab_path, merges_path)
    if not os.path.exists(tokenizer_json):
        bpe.save(tokenizer_json)

    return PreTrainedTokenizerFast(
        tokenizer_file=tokenizer_json,
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
    )


def resolve_trained_tokenizer_dirs(vocab_sizes_to_load):
    """Find tokenizer directories already written to Drive for the given vocab sizes."""
    resolved = {}
    for v_size in vocab_sizes_to_load:
        tok_dir = os.path.join(TOKENIZER_ROOT, f"bpe_{v_size}")
        if os.path.exists(tok_dir):
            resolved[v_size] = tok_dir
    return resolved

In [ ]:
# %%
# Define all Google Drive output locations used by the ablation study.
# Keeping these in one place makes it easy to redirect outputs later.

DRIVE_ROOT = "/content/drive/MyDrive/llm_folder"
ABLATION_ROOT = os.path.join(DRIVE_ROOT, "ablation_artifacts")
TOKENIZER_ROOT = os.path.join(ABLATION_ROOT, "tokenizers")
PROFILE_ROOT = os.path.join(ABLATION_ROOT, "profiles")
TOKENIZER_PACKED_ROOT = os.path.join(ABLATION_ROOT, "tokenizer_ablation_packed")
CORPUS_PACKED_ROOT = os.path.join(ABLATION_ROOT, "corpus_ablation_packed")
MODEL_ROOT = os.path.join(ABLATION_ROOT, "models")

for path in [ABLATION_ROOT, TOKENIZER_ROOT, PROFILE_ROOT, TOKENIZER_PACKED_ROOT, CORPUS_PACKED_ROOT, MODEL_ROOT]:
    os.makedirs(path, exist_ok=True)

print(f"Artifacts will be written under: {ABLATION_ROOT}")

## 1. Load and Inspect TinyStories in Colab

### Section 1 goal

These cells provide a quick sanity check that the dataset loads correctly and that the stories have the expected rough length before any tokenizer or training experiments begin.

In [ ]:
# %%
# Colab RAM optimization: reuse globally loaded `train_ds` instead of loading
# the dataset again in this section.

# Print the dataset schema and the first example to verify the text field layout.
print(train_ds)
print(train_ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2119719
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [ ]:
# %%
# Colab RAM optimization: compute quick word statistics from the global `train_ds`
# handle to avoid duplicating the dataset object in memory.

lengths = [len(example["text"].split()) for example in train_ds.select(range(1000))]
print("Avg words:", np.mean(lengths))
print("Max words:", np.max(lengths))

Avg words: 183.801
Max words: 837


## 2. Tokenizer Effect

This section automates the tokenizer ablation setup by training multiple byte-level BPE tokenizers on the same 100,000-story corpus and then profiling how vocabulary size changes sequence length on a fixed 5,000-story sample.

#### 2.1 Train custom tokenizers on 100,000 stories

This first step materializes a clean plain-text corpus file in Google Drive. Keeping it separate from tokenizer training makes it easy to rerun tokenizer sweeps without rewriting the corpus each time.

#### 2.1b Train and save the tokenizer family

In [ ]:
# %%
# Write a fixed 100k-story subset to a plain-text corpus file.
# The tokenizer trainer reads from this file, so we create it once and reuse it.

train_for_tokenizer = train_ds.select(range(min(100_000, len(train_ds))))
corpus_path = os.path.join(TOKENIZER_ROOT, "tiny_corpus_100k.txt")

with open(corpus_path, "w", encoding="utf-8") as f:
    for example in train_for_tokenizer:
        text = example.get("text", "")
        if text:
            # Normalize newline formatting so the corpus is consistent across platforms.
            cleaned = text.replace("\r\n", "\n").replace("\r", "\n").strip()
            if cleaned:
                f.write(cleaned + "\n")

print(f"Tokenizer corpus saved to: {corpus_path}")
print(f"Stories written: {len(train_for_tokenizer):,}")

In [ ]:
# %%
# Train five byte-level BPE tokenizers that differ only in vocabulary size.
# This isolates tokenizer capacity as the independent variable for Section 2.

trained_tokenizer_dirs = {}
for v_size in vocab_sizes:
    tok_dir = os.path.join(TOKENIZER_ROOT, f"bpe_{v_size}")
    os.makedirs(tok_dir, exist_ok=True)

    print(f"Training tokenizer with vocab_size={v_size} ...")
    bpe_tok = ByteLevelBPETokenizer()
    bpe_tok.train(
        files=[corpus_path],
        vocab_size=v_size,
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>"],
    )

    # Save both the vocab/merge pair and a unified tokenizer JSON for fast reloads.
    bpe_tok.save_model(tok_dir)
    bpe_tok.save(os.path.join(tok_dir, "tokenizer.json"))
    trained_tokenizer_dirs[v_size] = tok_dir

print(f"Saved tokenizer family under: {TOKENIZER_ROOT}")

#### 2.2 Profile average and maximum sequence lengths

After the tokenizer family is saved, this step encodes the same 5,000-story sample with each tokenizer and compares the resulting sequence lengths.

In [ ]:
# %%
# Profile tokenized sequence lengths on a random 5k-story sample.
# We report both the average and the maximum token count to show the trade-off
# between larger vocabularies and shorter encoded sequences.

trained_tokenizer_dirs = resolve_trained_tokenizer_dirs(vocab_sizes)
if len(trained_tokenizer_dirs) != len(vocab_sizes):
    raise FileNotFoundError("Some tokenizer folders are missing. Run Section 2.1 first.")

profile_sample = train_ds.shuffle(seed=SEED).select(range(min(5_000, len(train_ds))))
profile_rows = []

for v_size in vocab_sizes:
    tok_i = load_fast_tokenizer_from_folder(trained_tokenizer_dirs[v_size])

    # Encode the same sampled stories for every tokenizer to ensure a fair comparison.
    lengths = [len(tok_i(text, add_special_tokens=False)["input_ids"]) for text in profile_sample["text"]]
    profile_rows.append({
        "vocab_size": v_size,
        "avg_tokens_per_story": float(np.mean(lengths)),
        "max_tokens_per_story": int(np.max(lengths)),
    })

header = f"{'Vocab Size':>12} | {'Avg Tokens/Story':>18} | {'Max Tokens/Story':>18}"
print(header)
print("-" * len(header))
for row in profile_rows:
    print(f"{row['vocab_size']:>12,d} | {row['avg_tokens_per_story']:>18.2f} | {row['max_tokens_per_story']:>18,d}")

profile_path = os.path.join(PROFILE_ROOT, "tokenizer_length_profile.txt")
with open(profile_path, "w", encoding="utf-8") as f:
    f.write(header + "\n")
    f.write("-" * len(header) + "\n")
    for row in profile_rows:
        f.write(f"{row['vocab_size']:>12,d} | {row['avg_tokens_per_story']:>18.2f} | {row['max_tokens_per_story']:>18,d}\n")

print(f"\nSaved profile table to: {profile_path}")

## 3. Tokenizer Ablation Training

This section reuses the tokenizers from Section 2 and keeps the model architecture fixed. The only variable that changes across runs is the tokenizer itself, which makes validation loss directly comparable across vocabulary sizes.

In [ ]:
# %%
# Prepare the fixed 500k training subset and cache one packed dataset per tokenizer.
# This preprocessing step is separated from the training loop so failures or reruns
# do not force unnecessary retraining.

trained_tokenizer_dirs = resolve_trained_tokenizer_dirs(vocab_sizes)
if len(trained_tokenizer_dirs) != len(vocab_sizes):
    raise FileNotFoundError("Tokenizer folders are incomplete. Run Section 2.1 first.")

train_500k = train_ds.select(range(min(500_000, len(train_ds))))
val_fixed = val_ds.select(range(min(20_000, len(val_ds))))

for v_size in vocab_sizes:
    tok_i = load_fast_tokenizer_from_folder(trained_tokenizer_dirs[v_size])
    packed_dir = os.path.join(TOKENIZER_PACKED_ROOT, f"packed_vs{v_size}_bs{block_size}")

    if os.path.exists(packed_dir):
        print(f"[SKIP] Packed tokenizer dataset already exists: {packed_dir}")
        continue

    print(f"Caching packed dataset for vocab_size={v_size} ...")
    tokenize_and_pack_to_disk(
        train_subset=train_500k,
        val_subset=val_fixed,
        tokenizer=tok_i,
        out_dir=packed_dir,
        block_size=block_size,
    )

print("Tokenizer-specific packed datasets are ready.")

906933 9113


In [ ]:
# %%
# Train one fresh baseline model per tokenizer using the cached packed datasets.
# Login once in Colab before running this cell: !wandb login

trained_tokenizer_dirs = resolve_trained_tokenizer_dirs(vocab_sizes)
if len(trained_tokenizer_dirs) != len(vocab_sizes):
    raise FileNotFoundError("Tokenizer folders are incomplete. Run Section 2.1 first.")

for v_size in vocab_sizes:
    print(f"\n===== Tokenizer Ablation: vocab_size={v_size} =====")
    tok_i = None
    packed_ds = None
    lm_train_i = None
    lm_val_i = None
    model_i = None
    data_collator_i = None
    trainer_i = None

    try:
        tok_i = load_fast_tokenizer_from_folder(trained_tokenizer_dirs[v_size])
        packed_dir = os.path.join(TOKENIZER_PACKED_ROOT, f"packed_vs{v_size}_bs{block_size}")
        packed_ds = load_from_disk(packed_dir)

        lm_train_i = packed_ds["train"]
        lm_val_i = packed_ds["validation"]
        model_i = build_baseline_model(tok_i)
        data_collator_i = DataCollatorForLanguageModeling(tok_i, mlm=False)

        run_name = f"tok_vs_{v_size}"
        output_dir = os.path.join(MODEL_ROOT, "tokenizer_ablation", run_name)
        os.makedirs(output_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=2,
            eval_strategy="steps",
            eval_steps=500,
            logging_steps=100,
            save_steps=1000,
            save_total_limit=1,
            learning_rate=5e-4,
            warmup_steps=500,
            weight_decay=0.1,
            # We keep dataset size fixed in this study, so epoch-based training is fine.
            num_train_epochs=1,
            fp16=torch.cuda.is_available(),
            report_to=["wandb"],
            run_name=run_name,
            seed=SEED,
        )

        wandb.init(project="TinyStories-Tokenizer-Ablation", name=run_name)
        trainer_i = Trainer(
            model=model_i,
            args=training_args,
            train_dataset=lm_train_i,
            eval_dataset=lm_val_i,
            data_collator=data_collator_i,
        )
        trainer_i.train()
        trainer_i.save_model(output_dir)
        tok_i.save_pretrained(output_dir)
    except Exception as e:
        print(f"[ERROR] Tokenizer ablation failed for vocab_size={v_size}: {e}")
    finally:
        wandb.finish()
        del tok_i, packed_ds, lm_train_i, lm_val_i, model_i, data_collator_i, trainer_i
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Tokenizer ablation runs finished.")

With the tokenizer-ablation runs complete, the next section fixes the tokenizer at 8,192 vocabulary entries and changes only the corpus size to study scaling behavior under a constant compute budget.

## 4. Corpus Size Effect

This section switches the ablation axis from tokenizer choice to corpus size. The tokenizer is fixed at 8,192 vocabulary entries, and the compute budget is controlled later with a shared `max_steps` value.

In [ ]:
# %%
# Build and save packed datasets for each corpus size using the 8,192-vocab tokenizer.
# Saving tokenized subsets to disk prevents repeated preprocessing and avoids excess RAM use.

trained_tokenizer_dirs = resolve_trained_tokenizer_dirs(vocab_sizes)
if 8192 not in trained_tokenizer_dirs:
    raise FileNotFoundError("The 8,192-vocab tokenizer is missing. Run Section 2.1 first.")

tok_8192 = load_fast_tokenizer_from_folder(trained_tokenizer_dirs[8192])
val_fixed = val_ds.select(range(min(20_000, len(val_ds))))
corpus_subset_dirs = {}

for n in subset_sizes:
    n_eff = min(n, len(train_ds))
    out_dir = os.path.join(CORPUS_PACKED_ROOT, f"train_{n_eff}")

    if os.path.exists(out_dir):
        print(f"[SKIP] Using cached subset: {out_dir}")
        corpus_subset_dirs[n] = out_dir
        continue

    print(f"Packing subset size={n_eff:,} ...")
    train_subset_n = train_ds.select(range(n_eff))
    try:
        tokenize_and_pack_to_disk(
            train_subset=train_subset_n,
            val_subset=val_fixed,
            tokenizer=tok_8192,
            out_dir=out_dir,
            block_size=512,
        )
        corpus_subset_dirs[n] = out_dir
    except Exception as e:
        print(f"[ERROR] Failed for subset size {n_eff:,}: {e}")

print("Finished preparing corpus-size subsets.")

Total parameters: 6.918144 M


#### 4.1 Constant-compute training with max_steps

In [ ]:
# %%
# Train one 7M model per corpus subset with exactly the same number of optimizer steps.
# We use max_steps instead of num_train_epochs so every run consumes matched training compute.

trained_tokenizer_dirs = resolve_trained_tokenizer_dirs(vocab_sizes)
if 8192 not in trained_tokenizer_dirs:
    raise FileNotFoundError("The 8,192-vocab tokenizer is missing. Run Section 2.1 first.")

if not corpus_subset_dirs:
    for n in subset_sizes:
        n_eff = min(n, len(train_ds))
        candidate = os.path.join(CORPUS_PACKED_ROOT, f"train_{n_eff}")
        if os.path.exists(candidate):
            corpus_subset_dirs[n] = candidate

constant_max_steps = 10_000
tok_8192 = load_fast_tokenizer_from_folder(trained_tokenizer_dirs[8192])
corpus_runs_root = os.path.join(MODEL_ROOT, "corpus_ablation")
os.makedirs(corpus_runs_root, exist_ok=True)

for n in subset_sizes:
    n_eff = min(n, len(train_ds))
    packed_dir = corpus_subset_dirs.get(n)
    if packed_dir is None or not os.path.exists(packed_dir):
        print(f"[WARN] Missing packed data for subset size {n_eff:,}; skipping.")
        continue

    print(f"\n===== Corpus Ablation: train_size={n_eff:,}, max_steps={constant_max_steps} =====")
    packed_ds = None
    lm_train_n = None
    lm_val_n = None
    model_n = None
    data_collator_n = None
    trainer_n = None

    try:
        packed_ds = load_from_disk(packed_dir)
        lm_train_n = packed_ds["train"]
        lm_val_n = packed_ds["validation"]

        model_n = build_baseline_model(tok_8192)
        data_collator_n = DataCollatorForLanguageModeling(tok_8192, mlm=False)

        run_name = f"corpus_n_{n_eff}"
        output_dir = os.path.join(corpus_runs_root, run_name)
        os.makedirs(output_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=2,
            eval_strategy="steps",
            eval_steps=500,
            logging_steps=100,
            save_steps=1000,
            save_total_limit=1,
            learning_rate=5e-4,
            warmup_steps=500,
            weight_decay=0.1,
            # max_steps is the correct control knob for equal-compute comparisons.
            max_steps=constant_max_steps,
            # Trainer ignores epoch count once max_steps is set, but we keep it explicit for readability.
            num_train_epochs=1,
            fp16=torch.cuda.is_available(),
            report_to=["wandb"],
            run_name=run_name,
            seed=SEED,
        )

        wandb.init(project="TinyStories-Corpus-Ablation", name=run_name)
        trainer_n = Trainer(
            model=model_n,
            args=training_args,
            train_dataset=lm_train_n,
            eval_dataset=lm_val_n,
            data_collator=data_collator_n,
        )
        trainer_n.train()
        trainer_n.save_model(output_dir)
        tok_8192.save_pretrained(output_dir)
    except Exception as e:
        print(f"[ERROR] Corpus ablation failed for subset size {n_eff:,}: {e}")
    finally:
        wandb.finish()
        del packed_ds, lm_train_n, lm_val_n, model_n, data_collator_n, trainer_n
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Corpus-size ablation runs finished.")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,3.842734,3.673626
1000,3.006654,2.864876
1500,2.643113,2.514081
2000,2.449167,2.321668
2500,2.324586,2.210293
3000,2.245141,2.129375
3500,2.187576,2.076827
4000,2.132537,2.031412
4500,2.112084,1.992560
5000,2.079956,1.965953


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=28342, training_loss=1.996088468753394, metrics={'train_runtime': 2856.3162, 'train_samples_per_second': 317.518, 'train_steps_per_second': 9.923, 'total_flos': 1.3203519855132672e+16, 'train_loss': 1.996088468753394, 'epoch': 1.0})

## 5. Memorization Check on the 10k-Subset Model

The memorization check is easier to interpret when split into two steps: first generate a sample from the 10k-subset model, then score its n-gram overlap against the training corpus and log the results.

In [ ]:
# %%
# Load the trained 10k-subset model and generate one continuation for analysis.
# This generation is separated from scoring so you can inspect or replace the sample if needed.

memorization_n = 10_000
memorization_run_dir = os.path.join(MODEL_ROOT, "corpus_ablation", f"corpus_n_{memorization_n}")
if not os.path.exists(memorization_run_dir):
    raise FileNotFoundError(f"Trained 10k-subset model not found at: {memorization_run_dir}")

model_mem = GPT2LMHeadModel.from_pretrained(memorization_run_dir)
model_mem = model_mem.to("cuda" if torch.cuda.is_available() else "cpu")
model_mem.eval()

tok_mem = PreTrainedTokenizerFast.from_pretrained(memorization_run_dir)
prompt = "Once upon a time"
inputs = tok_mem(prompt, return_tensors="pt").to(model_mem.device)

with torch.no_grad():
    output = model_mem.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        top_p=0.95,
        temperature=0.9,
        eos_token_id=tok_mem.eos_token_id,
        pad_token_id=tok_mem.pad_token_id,
    )

full_text = tok_mem.decode(output[0], skip_special_tokens=True)
generated = full_text[len(prompt):].strip()

print("Prompt:", prompt)
print("\nGenerated continuation:\n", generated)

What is your name? This is my new dog. His name is Spot." The new dog barks and licks his face. Lily and the new dog's nose. They are very happy.

Sara and Ben play with the new dog. They throw the new dog and throw the new dog down. They make the new dog go over and over. They run and hug the new dog. They play with the new dog and the new dog. They are very happy.Lily and Ben were playing in the park. They saw a big tree with many leaves. They wanted to touch the leaves and the leaves. Mom said


The next cell applies `ngram_containment` at multiple n-gram sizes and logs the overlap metrics to WandB so the 10k-subset run can be compared with later memorization checks.

In [ ]:
# %%
# Score memorization by comparing generated n-grams against the TinyStories training set.
# Lower containment generally suggests less direct overlap with the training corpus.

wandb.init(project="TinyStories-Corpus-Ablation", name="memorization_check_10k")
for n in [4, 8]:
    stats = ngram_containment(
        generated_text=generated,
        tok=tok_mem,
        train_texts=train_ds["text"],
        n=n,
        max_train_docs=20_000,
        seed=SEED,
    )
    print(f"\nMemorization stats (n={n}):")
    print(stats)
    wandb.log({
        f"memorization/n{n}_containment": stats.get("containment", float("nan")),
        f"memorization/n{n}_overlap": stats.get("overlap_ngrams", 0),
        f"memorization/n{n}_gen_ngrams": stats.get("gen_ngrams", 0),
        f"memorization/n{n}_train_ngrams": stats.get("train_ngrams", 0),
    })
wandb.finish()

# Release GPU memory once the evaluation is done.
del model_mem
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()